# `AsVectorSpace`: Giving a manifold vector-space operations

Camera calibration can change over time: a zoom lens changes focal length, for example. GTSAM knows how to optimize a calibration as a **manifold**, but a cumulative spline expects values that can also be added and subtracted. `AsVectorSpace<Class>` is an explicit C++ adapter that supplies those operations in local coordinates.

This notebook develops that idea from first principles, shows a small calibration path, and explains when the approximation is—and is not—appropriate.

GTSAM Copyright 2010-2022, Georgia Tech Research Corporation,
Atlanta, Georgia 30332-0415
All Rights Reserved

Authors: Frank Dellaert, et al. (see THANKS for the full author list)

See LICENSE for the license information

<a href="https://colab.research.google.com/github/borglab/gtsam/blob/develop/gtsam/geometry/doc/AsVectorSpace.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install GTSAM from pip if running in Google Colab
try:
    import google.colab
    %pip install --quiet gtsam-develop plotly
except ImportError:
    pass  # Not in Colab

In [ ]:
import gtsam
import numpy as np
import plotly.graph_objects as go

## 1. Three structures, in plain language

These terms describe which operations are meaningful for a type:

| Structure | What you can do | GTSAM example |
|---|---|---|
| **Vector space** | Add values, subtract values, and multiply by a scalar. | `Vector5` |
| **Manifold** | Describe a small displacement in a local flat coordinate system, then move along that displacement. | `Cal3_S2` |
| **Lie group** | Compose and invert values, while also using local coordinates. | `Pose2`, `Pose3`, `Rot3` |

A camera calibration is naturally a manifold: changing $f_x$ by a small amount makes sense. It is not naturally a Lie group: multiplying or composing two intrinsic calibration matrices does not describe a physical operation.

## 2. The two manifold operations we need

GTSAM manifolds expose a pair of complementary operations:

- `localCoordinates(other)` answers: “what tangent vector moves me to `other`?”
- `retract(tangent)` answers: “where do I arrive after taking this local step?”

The round trip below turns a calibration into five local coordinates and reconstructs it. The default `Cal3_S2()` acts as the coordinate origin.

In [ ]:
origin = gtsam.Cal3_S2()
calibration = gtsam.Cal3_S2(500.0, 510.0, 0.0, 320.0, 240.0)
coordinates = origin.localCoordinates(calibration)
reconstructed = origin.retract(coordinates)

print("Coordinates [fx, fy, skew, px, py]:", coordinates)
print("Round trip succeeds:", reconstructed.equals(calibration, 1e-9))

## 3. The modeling choice made by `AsVectorSpace`

Choose the default-constructed value $e$ as an origin and write $\phi(x)=\operatorname{Local}(e,x)$. The adapter treats $\phi(x)$ as the vector representation of $x$. In that chosen chart, adding a tangent vector means retracting it from the current value, and adding two wrapped values means applying the second value's coordinates as a displacement:

$$x + v = \operatorname{Retract}(x,v), \qquad x + y = \operatorname{Retract}(x,\phi(y)).$$

Subtraction and negation use negated coordinates. This gives the interface needed by the spline while keeping the modeling decision visible in the type name.

> **Important:** the adapter does not discover a hidden physical group operation. It chooses a local affine model. Results can depend on the default value used as $e$, and large excursions can leave the region where this coordinate model is useful.

## 4. A small calibration path

To make the choice concrete, interpolate between two calibrations in coordinates about the default origin, then retract each intermediate coordinate vector. For `Cal3_S2`, this produces the simple focal-length path shown below. The code uses the wrapped Python manifold operations; the C++ adapter packages the same coordinate convention as operators.

In [ ]:
first = gtsam.Cal3_S2(500.0, 505.0, 0.0, 320.0, 240.0)
second = gtsam.Cal3_S2(650.0, 660.0, 0.0, 320.0, 240.0)
first_vector = origin.localCoordinates(first)
second_vector = origin.localCoordinates(second)

amounts = np.linspace(0.0, 1.0, 101)
path = [origin.retract((1.0 - a) * first_vector + a * second_vector) for a in amounts]

figure = go.Figure()
figure.add_scatter(x=amounts, y=[cal.fx() for cal in path], name="fx")
figure.add_scatter(x=amounts, y=[cal.fy() for cal in path], name="fy")
figure.update_layout(
    title="A local affine path between two calibrations",
    xaxis_title="interpolation amount",
    yaxis_title="focal length (pixels)",
    template="plotly_white",
)
figure.show()

## 5. C++ usage

`AsVectorSpace` is a header-only C++ template and is not currently exposed in the Python wrapper. It inherits the wrapped class, so ordinary calibration accessors remain available.

```cpp
using Calibration = AsVectorSpace<Cal3_S2>;
Calibration first(Cal3_S2(500.0, 505.0, 0.0, 320.0, 240.0));
Calibration second(Cal3_S2(650.0, 660.0, 0.0, 320.0, 240.0));
Calibration midpoint = first + 0.5 * (second - first).vector();
```

The explicit alias makes the approximation reviewable: readers can see that calibration is being treated as a local vector space rather than as a physical composition group.

## 6. Combining pose and calibration

A time-varying camera state may contain both a pose and a calibration. `CartesianProduct<A, B>` combines the two components:

```cpp
using CameraState = CartesianProduct<Pose3, AsVectorSpace<Cal3_S2>>;
CumulativeSplineTrajectory<CameraState> cameraTrajectory;
```

The pose component uses its natural Lie-group operations. The calibration component uses the explicit local affine model. The product lets the spline carry both together without claiming that camera intrinsics have a physical composition law.

## 7. A practical decision checklist

`AsVectorSpace` is a reasonable choice when all of the following are true:

- the wrapped type has meaningful local coordinates;
- the values stay near the chosen default-origin chart;
- addition in those coordinates matches the intended application model; and
- the approximation is documented as part of the model.

Prefer a domain-specific Lie group when composition has a physical meaning. Prefer direct manifold interpolation when no additive interpretation is appropriate. In either case, test the expected range of values rather than relying only on behavior near the origin.

## Source and related reading

- [AsVectorSpace.h](https://github.com/borglab/gtsam/blob/develop/gtsam/geometry/AsVectorSpace.h)
- [CartesianProduct.h](https://github.com/borglab/gtsam/blob/develop/gtsam/geometry/CartesianProduct.h)
- [Cumulative spline concepts](../../basis/doc/CumulativeSplineTrajectory.ipynb)
- [Runnable Pose2 spline example](../../../python/gtsam/examples/CumulativeSplineTrajectoryExample.ipynb)